# Session 4 · Exercises

*Course team · Licence: CC BY 4.0*

The exercises for the 30 minutes of student work in each block. Work in pairs; one of you types, the other directs.
The lecturer chooses which exercises to set; there are more than fit into 30 minutes. Stretch tasks are for pairs that finish
early.

Formats: *Paper* (pen and paper or whiteboard, no laptop), *Predict* (write down the result before you run any
code), *Code* (laptop), *Pairs* (one types, one directs), *Class* (the whole room together).

Data: Inside Airbnb, Berlin snapshot of 26 June 2026 (CC BY 4.0), prepared once with `uv run python case-study/prepare_airbnb.py`. Report results in aggregate and never try to identify a host.

## Part 1 · Data quality checks and missing values

Run the starter cell first: it loads the listings and the calendar.

In [ ]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

import numpy as np
import pandas as pd

listings = pd.read_parquet("case-study/data/airbnb/listings.parquet")
calendar = pd.read_parquet("case-study/data/airbnb/calendar.parquet")
print(listings.shape, calendar.shape)

### 1.1 · Parse ten messy prices

*Code, 15 min*

> `"$1,200.00"`, `"€80"`, `"80,50"`, `""`, `"N/A"`, `"0"`, `"99999"`, `" 75 "`, `"75.0"`, `"seventy"`. For each, decide: fix, mark as missing or flag.

*Source:* original exercise for this course; background: [`pandas.to_numeric`](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html)

In [ ]:
raw = pd.Series(["$1,200.00", "€80", "80,50", "", "N/A", "0", "99999", " 75 ", "75.0", "seventy"])
raw

### 1.2 · Sort the missingness cards

*Class, 10 min*

> Classify each scenario as MCAR, MAR or MNAR, and say which observed column (if any) explains the missing values.
>
> 1. A temperature sensor fails at random moments.
> 2. Younger respondents skip the income question more often than older ones.
> 3. Hosts of overpriced flats hide the price.
> 4. A survey page did not load for a random 5 % of visitors because of a server error.
> 5. In the Airbnb snapshot, the price is missing for almost every listing that has no free night in the next year.
> 6. Patients in a pain study who are in severe pain drop out before the final pain measurement.

*Source:* [van Buuren, *Flexible Imputation of Missing Data*, §1.2 MCAR, MAR and MNAR](https://stefvanbuuren.name/fimd/sec-MCAR.html); scenarios 1–3 from the course exercise bank, 4–6 original for this course

*Your answer:*

### 1.3 · What does mean imputation do?

*Predict, then Code, 10 min*

> Predict what happens to the variance and to the correlation with another column.
>
> Then check your prediction on the entire homes: fill the missing `bedrooms` with the mean and compare the variance of `bedrooms` and its correlation with `accommodates` before and after.

*Source:* [van Buuren, *Flexible Imputation of Missing Data*, §1.3 Simple solutions](https://stefvanbuuren.name/fimd/sec-simplesolutions.html)

*Your answer:*

### 1.4 · The same flat twice

*Code, 15 min*

> Find likely duplicates: same host, coordinates within about 20 metres, similar title.

*Source:* original exercise for this course; background: [Record linkage, Wikipedia](https://en.wikipedia.org/wiki/Record_linkage)

### 1.5 · Minimum stays in the calendar

*Code, 10 min*

> The calendar has a column `minimum_nights` for every listing and date. In how many calendar rows does it differ from the `minimum_nights` of the listing, and for how many listings? Is that a contradiction, or can a host set different minimum stays for different dates? What does it mean for a quality rule on minimum stays?

*Source:* original exercise for this course (Inside Airbnb Berlin case study)

### 1.6 · Are beds and bathrooms missing at random?

*Code, 15 min*

> About half of the listings have no value for `beds` and `bathrooms`. Compare the share of missing values by room type, for listings with and without a free night in the next year (`availability_365 == 0`) and for listings with and without a price. Which mechanism is plausible? Would you impute these two columns, and for which listings?

*Source:* original exercise for this course (Inside Airbnb Berlin case study)

## Part 2 · Outliers, transformations and a cleaning pipeline

Run the starter cell first: it loads the listings (if you start here) and adds the distance to the centre.

In [ ]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

import numpy as np
import pandas as pd

listings = pd.read_parquet("case-study/data/airbnb/listings.parquet")
listings["km_to_centre"] = 111.2 * np.hypot(listings["latitude"] - 52.5219,                     # Alexanderplatz
                                            (listings["longitude"] - 13.4132) * np.cos(np.radians(52.52)))
short = listings[listings["price"].notna() & listings["minimum_nights"].lt(28)].copy()   # comparable prices
print(len(listings), "listings,", len(short), "short-stay listings with a price")

### 2.1 · The z-score that can never fire

*Paper, 10 min*

> Ten prices, one of them extreme. Can its z-score exceed 3?

*Source:* [Shiffler (1988), "Maximum Z scores and outliers", *The American Statistician* 42(1)](https://doi.org/10.1080/00031305.1988.10475530)

*Your answer:*

### 2.2 · `log(0)`

*Predict, 3 min*

> What happens to free listings when the price is log-transformed?

*Source:* original exercise for this course; background: [`numpy.log1p`](https://numpy.org/doc/stable/reference/generated/numpy.log1p.html)

*Your answer:*

### 2.3 · The last digit of prices

*Predict, then Code, 10 min*

> How often should a price end in 0, 5 or 9? Predict the distribution, then plot it.

*Source:* original exercise for this course; background: [Psychological pricing, Wikipedia](https://en.wikipedia.org/wiki/Psychological_pricing)

*Your answer:*

### 2.4 · How many reviews per month is too many?

*Code, 15 min*

> Apply the IQR rule, the z-score (|z| > 3) and the MAD rule (|z\*| > 3.5) to `reviews_per_month`, first on the raw scale, then after `np.log1p`. Which share of listings does each rule flag? Look at the five largest values together with `number_of_reviews`, `number_of_reviews_ltm`, `first_review` and `room_type`: data errors or genuine values? Why is `reviews_per_month` missing for some listings, and should those be flagged?

*Source:* original exercise for this course (Inside Airbnb Berlin case study)

### 2.5 · An unusual price for the location

*Code, 15 min*

> For the entire homes in `short`, compute the classical and the robust (MCD) Mahalanobis distance on the log price and the log distance to the centre (`km_to_centre`). How many listings exceed the cut-off `scipy.stats.chi2.ppf(0.999, df=2)` with each method? Which kinds of listings are flagged? Which flags would you pass on to the data owner as questions, and which are genuine?

*Source:* original exercise for this course (Inside Airbnb Berlin case study)

### 2.6 · One more step for the cleaning pipeline (stretch)

*Code, 15 min, stretch task*

> Workbook 07 builds the cleaning pipeline from functions that each write one row of the cleaning log. Write a step `flag_bedrooms_above_guests(df)` that flags listings with more bedrooms than guests, records the rule, the number of rows affected, the action and the reason with `record`, and returns the new table. Run it on `listings`. How many listings does it flag, and which action do you choose?

*Source:* original exercise for this course (Inside Airbnb Berlin case study)

In [ ]:
log = []


def record(step, rule, affected, action, reason, df):
    """Append one row to the cleaning log and return the table unchanged (as in workbook 07)."""
    log.append({"step": step, "rule": rule, "rows_affected": int(affected), "action": action,
                "reason": reason, "rows_after": len(df)})
    return df